In [ ]:
# =============================================================================
# Feature Importance Comparison Visualization
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# =============================================================================
# Cell 1: Load Data
# =============================================================================

# Load the feature importance data
data_path = Path("../feature_importance.txt")
df = pd.read_csv(data_path, sep='\t')

print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
print(f"\nModels: {df['Model'].unique()}")
print(f"Styles: {df['Style'].unique()}")

# Preview
df.head(10)


In [ ]:
# =============================================================================
# Cell 2: Pivot to Wide Format
# =============================================================================

# Create pivot table: Rank x Model with Feature names as values
pivot_features = df.pivot(index='Rank', columns='Model', values='Feature')
pivot_styles = df.pivot(index='Rank', columns='Model', values='Style')

# Reorder columns to desired order
model_order = ["C6-Modis_Legacy", "XGBoost_Legacy", "XGBoost_Chipped", "Spatial-CNN_Chipped", "Pixel-Transformer_Chipped"]
pivot_features = pivot_features[[m for m in model_order if m in pivot_features.columns]]
pivot_styles = pivot_styles[[m for m in model_order if m in pivot_styles.columns]]

print("Features pivot shape:", pivot_features.shape)
print("\nFirst 10 rows:")
pivot_features.head(10)

In [ ]:
# =============================================================================
# Cell 3: Create Table Visualization (UPDATED - fixed gap and title overlap)
# =============================================================================

def create_feature_importance_table(pivot_features, pivot_styles, max_ranks=50, save_path=None):
    """
    Create a cleaner table-style visualization using matplotlib table.
    UPDATED: Larger text size, fixed gap and title overlap
    """
    
    pivot_features = pivot_features.head(max_ranks)
    pivot_styles = pivot_styles.head(max_ranks)
    
    n_ranks = len(pivot_features)
    n_models = len(pivot_features.columns)
    
    # Lighter colors
    color_modis = '#FFE4C4'      # Bisque (very light orange)
    color_pace = '#E0F0FF'       # Very light blue
    color_header = '#D0D0D0'     # Light gray for headers
    
    # Create figure
    fig, ax = plt.subplots(figsize=(36, 46))
    ax.axis('off')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    
    # Prepare table data
    table_data = []
    cell_colors = []
    
    # Header row
    headers = ['Rank'] + list(pivot_features.columns)
    header_colors = [color_header] * len(headers)
    
    for rank in pivot_features.index:
        row_data = [str(rank)]
        row_colors = [color_header]  # Rank column
        
        for model in pivot_features.columns:
            feature = pivot_features.loc[rank, model]
            style = pivot_styles.loc[rank, model]
            
            if pd.isna(feature):
                row_data.append('')
                row_colors.append('#F8F8F8')
            else:
                row_data.append(str(feature))
                if style == 'MODIS':
                    row_colors.append(color_modis)
                else:
                    row_colors.append(color_pace)
        
        table_data.append(row_data)
        cell_colors.append(row_colors)
    
    # Create table - position it below the title area
    table = ax.table(
        cellText=table_data,
        colLabels=headers,
        cellColours=cell_colors,
        colColours=header_colors,
        cellLoc='center',
        loc='bottom',
        bbox=[0.0, 0.0, 1.0, 1.10],  # [left, bottom, width, height] - leave 5% at top for title
        colWidths=[0.05] + [0.19] * n_models
    )
    
    # Style the table - BIGGER TEXT
    table.auto_set_font_size(False)
    table.set_fontsize(16)
    table.scale(1, 2.2)
    
    # Bold headers and rank column
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(fontweight='bold', fontsize=18)
        if col == 0:
            cell.set_text_props(fontweight='bold', fontsize=16)
    
    # Title - positioned at the very top
    fig.suptitle('Feature Importance Comparison Across Models - Top 50 Features by Rank', 
                 fontsize=20, fontweight='bold', y=0.99)
    
    # Legend - positioned at top right
    legend_elements = [
        mpatches.Patch(facecolor=color_modis, edgecolor='#999999', label='MODIS-style'),
        mpatches.Patch(facecolor=color_pace, edgecolor='#999999', label='PACE-style'),
    ]
    fig.legend(handles=legend_elements, loc='upper right', 
               fontsize=16, frameon=True, bbox_to_anchor=(0.98, 0.98))
    
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='white')
        print(f"Saved: {save_path}")
    
    plt.show()
    
    return fig


# Create the table visualization
fig_table = create_feature_importance_table(
    pivot_features,
    pivot_styles,
    max_ranks=50,
    save_path='feature_importance_table.png'
)


In [ ]:
# =============================================================================
# Cell 4: Summary Statistics
# =============================================================================

print("="*70)
print("SUMMARY: MODIS vs PACE Features by Model")
print("="*70)

summary_data = []

for model in model_order:
    if model not in df['Model'].values:
        continue
    
    model_df = df[df['Model'] == model]
    total = len(model_df)
    modis_count = len(model_df[model_df['Style'] == 'MODIS'])
    pace_count = len(model_df[model_df['Style'] == 'PACE'])
    
    summary_data.append({
        'Model': model,
        'Total': total,
        'MODIS': modis_count,
        'PACE': pace_count,
        'MODIS %': f"{100*modis_count/total:.1f}%",
        'PACE %': f"{100*pace_count/total:.1f}%"
    })
    
    print(f"\n{model}:")
    print(f"  Total features: {total}")
    print(f"  MODIS-style:    {modis_count} ({100*modis_count/total:.1f}%)")
    print(f"  PACE-style:     {pace_count} ({100*pace_count/total:.1f}%)")

summary_df = pd.DataFrame(summary_data)
print("\n")
print(summary_df.to_string(index=False))


In [ ]:
# =============================================================================
# Cell 5: Bar Chart - MODIS vs PACE by Model
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 6))

models = []
modis_counts = []
pace_counts = []

for model in model_order:
    if model not in df['Model'].values:
        continue
    
    model_df = df[df['Model'] == model]
    models.append(model.replace('_', '\n'))
    modis_counts.append(len(model_df[model_df['Style'] == 'MODIS']))
    pace_counts.append(len(model_df[model_df['Style'] == 'PACE']))

x = np.arange(len(models))
width = 0.35

# Lighter colors matching the grid
bars1 = ax.bar(x - width/2, modis_counts, width, label='MODIS-style', 
               color='#FFD9B3', edgecolor='#CC8800', linewidth=1.5)
bars2 = ax.bar(x + width/2, pace_counts, width, label='PACE-style', 
               color='#B3D9FF', edgecolor='#0066CC', linewidth=1.5)

ax.set_ylabel('Number of Features', fontsize=12)
ax.set_title('MODIS vs PACE Feature Distribution by Model', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0, 55)
ax.grid(axis='y', alpha=0.3)

# Add count labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{int(height)}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=11, fontweight='bold', color='#8B4513')

for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{int(height)}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=11, fontweight='bold', color='#00008B')

plt.tight_layout()
plt.savefig('modis_vs_pace_bar_chart.png', dpi=150, bbox_inches='tight')
print("Saved: modis_vs_pace_bar_chart.png")
plt.show()

In [ ]:
# =============================================================================
# Cell 5: Bar Chart - MODIS vs PACE by Model
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 6))

models = []
modis_counts = []
pace_counts = []

for model in model_order:
    if model not in df['Model'].values:
        continue
    
    model_df = df[df['Model'] == model]
    models.append(model.replace('_', '\n'))
    modis_counts.append(len(model_df[model_df['Style'] == 'MODIS']))
    pace_counts.append(len(model_df[model_df['Style'] == 'PACE']))

x = np.arange(len(models))
width = 0.35

# Lighter colors matching the grid
bars1 = ax.bar(x - width/2, modis_counts, width, label='MODIS-style', 
               color='#FFD9B3', edgecolor='#CC8800', linewidth=1.5)
bars2 = ax.bar(x + width/2, pace_counts, width, label='PACE-style', 
               color='#B3D9FF', edgecolor='#0066CC', linewidth=1.5)

ax.set_ylabel('Number of Features', fontsize=12)
ax.set_title('MODIS vs PACE Feature Distribution by Model', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0, 55)
ax.grid(axis='y', alpha=0.3)

# Add count labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{int(height)}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=11, fontweight='bold', color='#8B4513')

for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{int(height)}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=11, fontweight='bold', color='#00008B')

plt.tight_layout()
plt.savefig('modis_vs_pace_bar_chart.png', dpi=150, bbox_inches='tight')
print("Saved: modis_vs_pace_bar_chart.png")
plt.show()

In [ ]:
# =============================================================================
# Cell 6: Band/Index Distribution by Model
# =============================================================================

print("\n" + "="*70)
print("BAND/INDEX DISTRIBUTION BY MODEL")
print("="*70)

for model in model_order:
    if model not in df['Model'].values:
        continue
    
    model_df = df[df['Model'] == model]
    band_counts = model_df['Band/Index'].value_counts()
    
    print(f"\n{model}:")
    for band, count in band_counts.head(10).items():
        print(f"  {band:<15} {count:>3}")

In [ ]:
# =============================================================================
# Cell 7: Heatmap of Band Usage by Model (UPDATED - Greens colormap)
# =============================================================================

# Create a pivot of Band/Index counts by Model
band_model_counts = df.groupby(['Model', 'Band/Index']).size().unstack(fill_value=0)

# Reorder models
band_model_counts = band_model_counts.reindex(model_order)

# Get top bands (by total usage)
band_totals = band_model_counts.sum(axis=0).sort_values(ascending=False)
top_bands = band_totals.head(15).index.tolist()
band_model_counts = band_model_counts[top_bands]

fig, ax = plt.subplots(figsize=(16, 7))

# UPDATED: Use Greens colormap
im = ax.imshow(band_model_counts.values, cmap='Greens', aspect='auto', vmin=0)

# Labels
ax.set_xticks(range(len(top_bands)))
ax.set_xticklabels(top_bands, rotation=45, ha='right', fontsize=11)
ax.set_yticks(range(len(model_order)))
ax.set_yticklabels([m.replace('_', ' ') for m in model_order], fontsize=11)

# Add count annotations
for i in range(len(model_order)):
    for j in range(len(top_bands)):
        val = band_model_counts.values[i, j]
        # Adjust text color for green colormap
        text_color = 'white' if val > band_model_counts.values.max() * 0.5 else 'black'
        ax.text(j, i, str(int(val)), ha='center', va='center', 
               fontsize=10, color=text_color, fontweight='bold')

cbar = plt.colorbar(im, ax=ax, label='Feature Count', shrink=0.8)
cbar.ax.tick_params(labelsize=10)

ax.set_title('Band/Index Usage Across Models (Top 15)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Band/Index', fontsize=12)
ax.set_ylabel('Model', fontsize=12)

plt.tight_layout()
plt.savefig('band_usage_heatmap.png', dpi=150, bbox_inches='tight')
print("Saved: band_usage_heatmap.png")
plt.show()

In [ ]:




# =============================================================================
# Cell 8: Stacked Bar - Band Distribution per Model (UPDATED - tab10 colormap, fixed legend order)
# =============================================================================

# Categorize bands
def categorize_band(band):
    if band in ['Band_1', 'Band_2', 'Band_3', 'Band_4', 'Band_5', 'Band_6', 'Band_7']:
        return 'MODIS Optical'
    elif band in ['Band31', 'Snow']:
        return 'MODIS Thermal/Snow'
    elif band == 'NDVI':
        return 'NDVI'
    elif band in ['CCI', 'MTCI', 'CIRE', 'GCI']:
        return 'Chlorophyll Indices'
    elif band in ['REP', 'REIP', 'NDRE']:
        return 'Red Edge'
    elif band in ['EVI', 'PRI', 'mARI']:
        return 'Vegetation Stress'
    elif band in ['NDWI', 'NDII']:
        return 'Water Indices'
    elif band == 'Phenology':
        return 'Phenology'
    else:
        return 'Other'

df['Category'] = df['Band/Index'].apply(categorize_band)

# Create stacked bar chart
categories = ['MODIS Optical', 'MODIS Thermal/Snow', 'NDVI', 'Chlorophyll Indices', 
              'Red Edge', 'Vegetation Stress', 'Water Indices', 'Phenology', 'Other']

# Use tab10 colormap for distinguishable colors
tab10 = plt.cm.tab10
category_colors = {
    'MODIS Optical': tab10(0),       # Blue
    'MODIS Thermal/Snow': tab10(1),  # Orange
    'NDVI': tab10(2),                # Green
    'Chlorophyll Indices': tab10(3), # Red
    'Red Edge': tab10(4),            # Purple
    'Vegetation Stress': tab10(5),   # Brown
    'Water Indices': tab10(6),       # Pink
    'Phenology': tab10(7),           # Gray
    'Other': tab10(8)                # Olive
}

fig, ax = plt.subplots(figsize=(14, 8))

bottom = np.zeros(len(model_order))
x = np.arange(len(model_order))

# Store handles and labels for legend
handles = []
labels = []

for category in categories:
    counts = []
    for model in model_order:
        model_df = df[(df['Model'] == model) & (df['Category'] == category)]
        counts.append(len(model_df))
    
    if sum(counts) > 0:  # Only plot if category has data
        bar = ax.bar(x, counts, bottom=bottom, label=category, 
               color=category_colors[category], edgecolor='white', linewidth=0.5)
        handles.append(bar)
        labels.append(category)
        bottom += counts

ax.set_xticks(x)
ax.set_xticklabels([m.replace('_', '\n') for m in model_order], fontsize=11)
ax.set_ylabel('Number of Features', fontsize=12)
ax.set_title('Feature Category Distribution by Model', fontsize=14, fontweight='bold')

# Reverse the legend order to match visual stacking (bottom to top)
ax.legend(handles[::-1], labels[::-1], loc='upper right', bbox_to_anchor=(1.25, 1.0), fontsize=10)

ax.set_ylim(0, 55)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('feature_category_stacked_bar.png', dpi=150, bbox_inches='tight')
print("Saved: feature_category_stacked_bar.png")
plt.show()

In [ ]:
# =============================================================================
# Cell 9: Bump Chart - Feature Rank Changes Across Models
# =============================================================================

def create_bump_chart(df, max_rank=30, min_models=1, save_path=None):
    """
    Create a bump chart showing how features change rank across models.
    Features are shown as labeled boxes connected by lines.

    min_models: only draw features that appear in the top max_rank for at
    least this many models (default 1 = no filtering, original behavior).
    Raising this to 2+ drops single-model-only features, which is most of
    what makes the full chart busy.
    """
    
    # Get unique features that appear in top max_rank for any model
    top_features = set()
    for model in model_order:
        model_df = df[(df['Model'] == model) & (df['Rank'] <= max_rank)]
        top_features.update(model_df['Feature'].tolist())
    
    # Create a color map for Band/Index categories
    all_bands = df['Band/Index'].unique()
    tab20 = plt.cm.tab20
    band_colors = {band: tab20(i % 20) for i, band in enumerate(sorted(all_bands))}
    
    # Create figure
    fig, ax = plt.subplots(figsize=(30, 38))
    
    # X positions for each model
    x_positions = {model: i * 4.0 for i, model in enumerate(model_order)}
    
    # Track which features appear in which models and at what rank
    feature_positions = {}
    for feature in top_features:
        feature_positions[feature] = {}
        for model in model_order:
            model_df = df[(df['Model'] == model) & (df['Feature'] == feature)]
            if len(model_df) > 0:
                rank = model_df['Rank'].values[0]
                if rank <= max_rank:
                    feature_positions[feature][model] = rank
    
    # Keep only features shared across at least min_models columns
    feature_positions = {f: pos for f, pos in feature_positions.items() if len(pos) >= min_models}
    top_features = set(feature_positions.keys())
    
    # Draw connections and boxes
    for feature, positions in feature_positions.items():
        if len(positions) < 1:
            continue
        
        # Get band/index for color
        feature_df = df[df['Feature'] == feature].iloc[0]
        band = feature_df['Band/Index']
        color = band_colors[band]
        
        # Get ordered models where this feature appears
        models_with_feature = [m for m in model_order if m in positions]
        
        # Draw connecting lines
        if len(models_with_feature) > 1:
            x_coords = [x_positions[m] for m in models_with_feature]
            y_coords = [max_rank + 1 - positions[m] for m in models_with_feature]
            ax.plot(x_coords, y_coords, color=color, alpha=0.6, linewidth=4, zorder=1)
        
        # Draw boxes with labels
        for model in models_with_feature:
            x = x_positions[model]
            y = max_rank + 1 - positions[model]
            
            # Full feature name (truncate only if very long)
            display_name = feature
            if len(display_name) > 35:
                display_name = display_name[:32] + '...'
            
            # Draw box - MUCH BIGGER
            bbox = dict(
                boxstyle='round,pad=0.8,rounding_size=0.3', 
                facecolor=color, 
                edgecolor='white', 
                alpha=0.95, 
                linewidth=2
            )
            text_color = 'white' if sum(color[:3]) < 1.8 else 'black'
            ax.text(x, y, display_name, ha='center', va='center', 
                   fontsize=16, fontweight='bold',
                   bbox=bbox, color=text_color, zorder=2)
    
    # Styling
    ax.set_xlim(-2, max(x_positions.values()) + 2)
    ax.set_ylim(-0.5, max_rank + 2)
    
    # X-axis labels
    ax.set_xticks([x_positions[m] for m in model_order])
    ax.set_xticklabels([m.replace('_', '\n') for m in model_order], fontsize=20, fontweight='bold')
    
    # Y-axis (rank)
    ax.set_yticks(range(1, max_rank + 1))
    ax.set_yticklabels([str(max_rank + 1 - i) for i in range(1, max_rank + 1)], fontsize=14)
    ax.set_ylabel('Rank', fontsize=22, fontweight='bold')
    
    title_suffix = f' -- Shared in >= {min_models} Models' if min_models > 1 else ''
    ax.set_title(f'Feature Rank Changes Across Models (Top 30){title_suffix}', fontsize=28, fontweight='bold', pad=20)
    
    # Add gridlines
    ax.grid(axis='y', alpha=0.2, linestyle='-', color='gray')
    ax.set_axisbelow(True)
    
    # Remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Create legend for Band/Index colors
    used_bands = set()
    for feature in top_features:
        feature_df = df[df['Feature'] == feature]
        if len(feature_df) > 0:
            used_bands.add(feature_df.iloc[0]['Band/Index'])
    
    legend_handles = [mpatches.Patch(facecolor=band_colors[band], edgecolor='white', label=band) 
                      for band in sorted(used_bands)]
    
    ax.legend(handles=legend_handles, loc='center left', bbox_to_anchor=(1.01, 0.5), 
              fontsize=16, title='Band/Index', title_fontsize=18, frameon=True,
              fancybox=True, shadow=True)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='white')
        print(f"Saved: {save_path}")
    
    plt.show()
    return fig


# Create bump chart
fig_bump = create_bump_chart(df, max_rank=30, save_path='feature_bump_chart.png')

In [ ]:
# =============================================================================
# Cell 9b: Bump Chart -- Shared Features Only (less busy)
# =============================================================================
# Same chart as Cell 9, but only draws features that show up in the top-30
# for 2 or more models -- drops the single-model-only features that made
# the full chart hard to read.

fig_bump_shared = create_bump_chart(df, max_rank=30, min_models=2, save_path='feature_bump_chart_shared.png')


In [ ]:
# =============================================================================
# Cell 10: Table with Color Coding by Band/Index (UPDATED - fixed title overlap)
# =============================================================================

def create_band_colored_table(pivot_features, df, max_ranks=50, save_path=None):
    """
    Create a table visualization with cells colored by Band/Index category.
    """
    
    pivot_features = pivot_features.head(max_ranks)
    
    n_ranks = len(pivot_features)
    n_models = len(pivot_features.columns)
    
    # Create color map for Band/Index
    all_bands = df['Band/Index'].unique()
    tab20 = plt.cm.tab20
    band_colors = {band: tab20(i % 20) for i, band in enumerate(sorted(all_bands))}
    
    # Header color
    color_header = '#D0D0D0'
    
    # Create figure
    fig, ax = plt.subplots(figsize=(40, 52))
    ax.axis('off')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    
    # Prepare table data
    table_data = []
    cell_colors = []
    
    for rank in pivot_features.index:
        row_data = [str(rank)]
        row_colors = [color_header]  # Rank column
        
        for model in pivot_features.columns:
            feature = pivot_features.loc[rank, model]
            
            if pd.isna(feature):
                row_data.append('')
                row_colors.append('#F8F8F8')
            else:
                row_data.append(str(feature))
                # Get band/index for this feature
                feature_df = df[(df['Model'] == model) & (df['Feature'] == feature)]
                if len(feature_df) > 0:
                    band = feature_df.iloc[0]['Band/Index']
                    row_colors.append(band_colors[band])
                else:
                    row_colors.append('#F8F8F8')
        
        table_data.append(row_data)
        cell_colors.append(row_colors)
    
    # Headers
    headers = ['Rank'] + list(pivot_features.columns)
    header_colors = [color_header] * len(headers)
    
    # Create table - position it below the title area
    table = ax.table(
        cellText=table_data,
        colLabels=headers,
        cellColours=cell_colors,
        colColours=header_colors,
        cellLoc='center',
        loc='bottom',
        bbox=[0.0, 0.0, 1.0, 1.10],  # [left, bottom, width, height] - leave 5% at top for title
        colWidths=[0.04] + [0.192] * n_models
    )
    
    # Style the table - LARGE TEXT (fontsize 16)
    table.auto_set_font_size(False)
    table.set_fontsize(16)
    table.scale(1, 2.8)  # Taller rows for bigger text
    
    # Style headers and cells
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(fontweight='bold', fontsize=18)
        if col == 0:
            cell.set_text_props(fontweight='bold', fontsize=16)
        
        # Adjust text color based on background brightness
        if row > 0 and col > 0:
            bg_color = cell_colors[row - 1][col]
            if isinstance(bg_color, tuple):
                brightness = sum(bg_color[:3]) / 3
                if brightness < 0.5:
                    cell.set_text_props(color='white')
                else:
                    cell.set_text_props(color='black')
    
    # Title - positioned at the very top
    fig.suptitle('Feature Importance by Model - Colored by Band/Index', 
                 fontsize=24, fontweight='bold', y=0.99)
    
    # Create legend for Band/Index colors
    used_bands = set()
    for model in pivot_features.columns:
        for rank in pivot_features.index:
            feature = pivot_features.loc[rank, model]
            if pd.notna(feature):
                feature_df = df[(df['Model'] == model) & (df['Feature'] == feature)]
                if len(feature_df) > 0:
                    used_bands.add(feature_df.iloc[0]['Band/Index'])
    
    legend_handles = [mpatches.Patch(facecolor=band_colors[band], edgecolor='black', label=band) 
                      for band in sorted(used_bands)]
    
    fig.legend(handles=legend_handles, loc='upper right', bbox_to_anchor=(0.98, 0.98), 
               fontsize=14, title='Band/Index', title_fontsize=16, ncol=2)
    
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='white')
        print(f"Saved: {save_path}")
    
    plt.show()
    return fig


# Create band-colored table
fig_band_table = create_band_colored_table(pivot_features, df, max_ranks=50, 
                                            save_path='feature_importance_by_band.png')

In [ ]:
# =============================================================================
# Cell: Create Highlighted Feature Importance Tables (One per Model)
# =============================================================================

def create_highlighted_feature_table(pivot_features, pivot_styles, highlight_model, max_ranks=50, save_path=None):
    """
    Create a table visualization highlighting features from one model.
    - Highlight model column: all features in light yellow
    - Other columns: matching features in light yellow, non-matching in light blue
    - Shows count of matching features above table
    """
    
    pivot_features = pivot_features.head(max_ranks)
    pivot_styles = pivot_styles.head(max_ranks)
    
    n_ranks = len(pivot_features)
    n_models = len(pivot_features.columns)
    
    # Colors
    color_match = '#FFFACD'       # Light yellow (LemonChiffon)
    color_nomatch = '#E0F0FF'     # Light blue
    color_header = '#D0D0D0'      # Light gray for headers
    color_highlight_header = '#FFD700'  # Gold for highlighted model header
    
    # Get features from the highlight model
    highlight_features = set(pivot_features[highlight_model].dropna().values)
    
    # Calculate matching counts for each other model
    match_counts = {}
    for model in pivot_features.columns:
        if model != highlight_model:
            model_features = set(pivot_features[model].dropna().values)
            match_counts[model] = len(highlight_features.intersection(model_features))
    
    # Create figure
    fig, ax = plt.subplots(figsize=(36, 48))
    ax.axis('off')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    
    # Prepare table data
    table_data = []
    cell_colors = []
    
    # Header row colors
    header_colors = [color_header]
    for model in pivot_features.columns:
        if model == highlight_model:
            header_colors.append(color_highlight_header)
        else:
            header_colors.append(color_header)
    
    headers = ['Rank'] + list(pivot_features.columns)
    
    for rank in pivot_features.index:
        row_data = [str(rank)]
        row_colors = [color_header]  # Rank column
        
        for model in pivot_features.columns:
            feature = pivot_features.loc[rank, model]
            
            if pd.isna(feature):
                row_data.append('')
                row_colors.append('#F8F8F8')
            else:
                row_data.append(str(feature))
                
                if model == highlight_model:
                    # Highlight model - always yellow
                    row_colors.append(color_match)
                else:
                    # Other models - yellow if matches, blue if not
                    if feature in highlight_features:
                        row_colors.append(color_match)
                    else:
                        row_colors.append(color_nomatch)
        
        table_data.append(row_data)
        cell_colors.append(row_colors)
    
    # Create table
    table = ax.table(
        cellText=table_data,
        colLabels=headers,
        cellColours=cell_colors,
        colColours=header_colors,
        cellLoc='center',
        loc='bottom',
        bbox=[0.0, 0.0, 1.0, 1.05],
        colWidths=[0.05] + [0.19] * n_models
    )
    
    # Style the table
    table.auto_set_font_size(False)
    table.set_fontsize(16)
    table.scale(1, 2.2)
    
    # Bold headers and rank column
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(fontweight='bold', fontsize=18)
            # Extra bold for highlight model
            if col > 0 and list(pivot_features.columns)[col-1] == highlight_model:
                cell.set_text_props(fontweight='bold', fontsize=20)
        if col == 0:
            cell.set_text_props(fontweight='bold', fontsize=16)
    
    # Title
    fig.suptitle(f'Feature Importance: Highlighting {highlight_model}', 
                 fontsize=24, fontweight='bold', y=0.995)
    
    # Match counts subtitle
    match_text = "Matching features: " + " | ".join([f"{model}: {count}" for model, count in match_counts.items()])
    fig.text(0.5, 0.975, match_text, ha='center', fontsize=16, style='italic')
    
    # Legend
    legend_elements = [
        mpatches.Patch(facecolor=color_match, edgecolor='#999999', label=f'In {highlight_model}'),
        mpatches.Patch(facecolor=color_nomatch, edgecolor='#999999', label=f'Not in {highlight_model}'),
    ]
    fig.legend(handles=legend_elements, loc='upper right', 
               fontsize=16, frameon=True, bbox_to_anchor=(0.98, 0.98))
    
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='white')
        print(f"Saved: {save_path}")
    
    plt.show()
    
    return fig, match_counts


# =============================================================================
# Cell: Generate All 6 Highlighted Tables
# =============================================================================

# List of all models
all_models = ["MODIS-C6", "XGBoost-C6_Chips", "XGBoost-LegacyData", "CNN-C6_Chips", "MLP-C6_Chips", "Transformer-C6_Chips"]

# Store all match counts for summary
all_match_counts = {}

# Generate a highlighted table for each model
for model in all_models:
    if model in pivot_features.columns:
        print(f"\n{'='*60}")
        print(f"Generating table for: {model}")
        print('='*60)
        
        fig, match_counts = create_highlighted_feature_table(
            pivot_features,
            pivot_styles,
            highlight_model=model,
            max_ranks=50,
            save_path=f'feature_importance_highlight_{model.replace("-", "_")}.png'
        )
        
        all_match_counts[model] = match_counts
        plt.close(fig)  # Close to free memory
    else:
        print(f"Warning: {model} not found in pivot_features columns")


# =============================================================================
# Cell: Summary of Feature Overlap (Bonus Heatmap!)
# =============================================================================

def create_overlap_summary(all_match_counts, save_path=None):
    """
    Create a summary heatmap of feature overlap between models.
    """
    models = list(all_match_counts.keys())
    n_models = len(models)
    
    # Create overlap matrix
    overlap_matrix = np.zeros((n_models, n_models))
    
    for i, model1 in enumerate(models):
        for j, model2 in enumerate(models):
            if model1 == model2:
                overlap_matrix[i, j] = 50  # Max features (self)
            elif model2 in all_match_counts[model1]:
                overlap_matrix[i, j] = all_match_counts[model1][model2]
    
    # Create heatmap
    fig, ax = plt.subplots(figsize=(12, 10))
    
    im = ax.imshow(overlap_matrix, cmap='YlOrRd', aspect='auto')
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Number of Matching Features', fontsize=12)
    
    # Set ticks
    ax.set_xticks(range(n_models))
    ax.set_yticks(range(n_models))
    ax.set_xticklabels(models, rotation=45, ha='right', fontsize=11)
    ax.set_yticklabels(models, fontsize=11)
    
    # Add text annotations
    for i in range(n_models):
        for j in range(n_models):
            value = int(overlap_matrix[i, j])
            text_color = 'white' if value > 25 else 'black'
            ax.text(j, i, str(value), ha='center', va='center', 
                    fontsize=14, fontweight='bold', color=text_color)
    
    ax.set_title('Feature Overlap Matrix\n(Number of shared features in Top 50)', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('Model', fontsize=12)
    ax.set_ylabel('Reference Model', fontsize=12)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='white')
        print(f"Saved: {save_path}")
    
    plt.show()
    
    return fig

# Create the overlap summary heatmap
fig_summary = create_overlap_summary(
    all_match_counts,
    save_path='feature_overlap_summary.png'
)